# 00 Project Summary

This notebook is a read-only dashboard for a generic M/EEG BIDS project.

It summarizes:

- available BIDS entities
- source recordings in the configured `sourcedata_root`
- raw BIDS conversion status
- raw `events.tsv` status and basic event QC
- derivative analysis-event status
- manually saved bad-channel decisions
- filtered raw derivative status
- bad-segment annotation status
- ICA, cleaned raw, epoch, and evoked derivative status
- existing files in `derivatives/meeg-pipeline/`

It should not create, overwrite, or modify project files.


## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import (
    bad_annotations_status_to_dataframe,
    bad_channels_status_to_dataframe,
    bids_entities_to_dataframe,
    derivative_files_to_dataframe,
    event_qc_to_dataframe,
    iter_recordings,
    project_status_summary_to_dataframe,
    raw_events_status_to_dataframe,
    selected_recordings_to_dataframe,
    source_recordings_to_dataframe,
    summary_derivative_paths_to_dataframe,
    summary_derivative_status_to_dataframe,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## Selection

Use single values, lists, `None`, or `"all"`.

Examples:

```python
SUBJECTS = "001"
SUBJECTS = ["001", "002"]
SUBJECTS = "all"

TASKS = "rest"
TASKS = ["auditory", "visual"]
TASKS = "all"
TASKS = ["all"]
```

If the project has no sessions or runs, keep `SESSIONS = None` and `RUNS = None`.


In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"
selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## BIDS entities

In [ ]:
bids_entities_to_dataframe(config)


## Source recordings in `sourcedata/`

This section checks whether the standardized `sourcedata/` import structure is present and what BIDS targets it maps to.


In [ ]:
source_recordings_table = source_recordings_to_dataframe(config)
source_recordings_table


## Raw BIDS and events.tsv status

In [ ]:
raw_events_status = raw_events_status_to_dataframe(config, selected_recordings)
raw_events_status


## Event QC overview

This reads existing `events.tsv` files and summarizes basic timing properties.

For event streams that should occur at regular intervals, the onset-difference columns are useful sanity checks.


In [ ]:
event_qc = event_qc_to_dataframe(config, selected_recordings)
event_qc


## Bad-channel overview

This checks whether a manual bad-channel decision exists for each selected recording.

Bad-channel decisions are usually stored in the step-specific QC derivative folder:

```text
derivatives/meeg-pipeline/sub-*/.../qc/*_desc-badchannels.json
```


In [ ]:
bad_channel_status = bad_channels_status_to_dataframe(
    config,
    selected_recordings,
)

bad_channel_status


## Bad-segment annotation overview

This checks whether manually saved `BAD*` time-segment annotations exist for each selected recording.

Bad-segment annotation files are usually stored in the step-specific cleaning derivative folder:

```text
derivatives/meeg-pipeline/sub-*/.../cleaning/*_desc-badsegments_annotations.fif
```


In [ ]:
annotation_status = bad_annotations_status_to_dataframe(
    config,
    selected_recordings,
)

annotation_status


## Expected derivative status matrix

This is a compact pipeline-status matrix.

It includes generic pipeline outputs such as:

- bad-channel decisions
- filtered continuous data
- bad-segment annotation derivatives
- ICA files and ICA decisions
- cleaned continuous data
- derivative analysis events
- epochs
- evokeds

The matrix is intentionally project-independent: it checks files and basic availability, but does not interpret project-specific event semantics.


In [ ]:
derivative_matrix = summary_derivative_status_to_dataframe(
    config,
    selected_recordings,
)

derivative_matrix


## Expected derivative paths

This shows the paths behind the status matrix above.


In [ ]:
expected_paths = summary_derivative_paths_to_dataframe(
    config,
    selected_recordings,
)

expected_paths


## Existing derivative files

This lists all files currently found under `derivatives/meeg-pipeline/`.

It is useful for checking whether unexpected or old outputs are present.


In [ ]:
derivatives_table = derivative_files_to_dataframe(config)
derivatives_table


## Project status summary

A compact, high-level status overview.


In [ ]:
project_status_summary_to_dataframe(
    selected_recordings=selected_recordings,
    source_recordings=source_recordings_table,
    raw_events_status=raw_events_status,
    bad_channel_status=bad_channel_status,
    annotation_status=annotation_status,
    derivative_matrix=derivative_matrix,
    derivative_files=derivatives_table,
)
